# 13. A reproducible terrain investigation

**Atlas source reading copy · reference-only**

[Course guide](index.ipynb) · [Pinned original](https://github.com/jltobias/JupyterLite-terrio/blob/18574d4e7a8dabb99d3db65b20c6180185578482/notebooks/13-capstone.ipynb)



Reading/editing copy only in the shared Atlas kernel. TerrIO 0.9.0 uses a pyemscripten_2025_0 wasm wheel for Pyodide 0.29; the source pins jupyterlite-pyodide-kernel 0.8.0b0 (default Pyodide 0.29.4). This wheel is incompatible with Atlas Pyodide 314; do not install it here or downgrade the shared runtime. [Run this lesson in its matched source Lab](https://jltobias.github.io/JupyterLite-terrio/lite/lab/index.html?path=13-capstone.ipynb). The source uses Plotly 6.3.1, xarray 2025.10.1 and optional CDN/WebGL graphics. Natural Earth countries are real generalized geography; all other numerical fixtures are explicitly synthetic. Native HTTP range I/O, specialized drivers and real service/model calls remain optional external continuations.

**Provenance:** `notebooks/13-capstone.ipynb` at `18574d4e7a8dabb99d3db65b20c6180185578482`; original SHA-256 `bb53ef464d228fc4f0a55f08f78864e421cb6f078c139c902f950c30a3b876bc`. Original code MIT; original teaching prose, synthetic data and chapter SVG diagrams CC0. Natural Earth is public domain; TerrIO/dependency/EPSG terms remain separately scoped in retained notices. No TerrIO wheel redistributed.

**Verification:** imported by static inspection; saved upstream outputs cleared. Runtime compatibility is separate from executed verification. See the course guide for dependencies, excluded assets, aliases and original credits.


![Chapter 13: A reproducible terrain investigation; abstract contour illustration](assets/13-capstone.svg)

# 13. A reproducible terrain investigation

From question to evidence

**Learning goals:** Combine terrain analysis, threshold sensitivity, 3D inspection, exported products, and a concise cited report.

**Time:** 30–45 minutes. **Prerequisites:** basic Python arrays; earlier chapters are helpful but every notebook runs independently.

**Run:** choose the Python (Pyodide) kernel in JupyterLite, then **Run → Run All Cells**. First use downloads the runtime and packages. Native Python users install the repository requirements first. Plotly views need network access to its versioned JavaScript CDN and 3D views need WebGL; static figures and numerical summaries remain available.

[Open this chapter in JupyterLite](https://jltobias.github.io/JupyterLite-terrio/lite/lab/index.html?path=13-capstone.ipynb) · [Learning resources](https://jltobias.github.io/JupyterLite-terrio/book/resources.html)

**Data:** all generated measurements are synthetic. Natural Earth outlines, where used, are public domain. Generated files go to `outputs/`; download them to retain them outside this browser.


In [ ]:
import sys
if sys.platform == "emscripten":
    import piplite
    await piplite.install(["terrio==0.9.0", "matplotlib", "plotly==6.3.1", "xarray==2025.10.1"])
import json
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import terrio
from terrio_lab import (DATA, OUTPUT, CRS, TRANSFORM, CREDIT, terrain, raster,
                        countries, rings, map_outlines, image_grid, interactive, surface, globe)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
print("Ready: TerrIO", terrio.__version__, "|", "browser" if sys.platform == "emscripten" else "native Python")

## Define a modest question

In this invented landscape, what fraction of valid land is above 1500 m and has slope below 12°? This is a demonstration of a reproducible spatial query. It is not a real site-selection, flood-risk, habitat, or engineering assessment.

In [ ]:
dem = raster(); z = dem.read(1)
slope = terrio.dem(dem, "slope").read(1)
valid = np.isfinite(slope) & (slope >= 0) & (slope <= 90)
high, gentle = 1500, 12
selected = valid & (z > high) & (slope < gentle)
fraction = selected.sum()/valid.sum()
area_km2 = selected.sum()*30*30/1e6
print(f"Selected: {fraction:.1%} of valid pixels; projected grid area: {area_km2:.3f} km²")
assert 0 <= fraction <= 1

## Show the evidence

Report the two inputs and the decision mask. Edge pixels with missing slope are excluded from the denominator. The selected area is a planar grid estimate, not an ellipsoidal land-area calculation.

In [ ]:
image_grid([z], ["Elevation input"]); plt.show()
image_grid([slope], ["Slope input"], ["magma"], units="degrees"); plt.show()
image_grid([np.where(valid, selected.astype(float), np.nan)], ["Selected = 1; excluded = 0"], ["viridis"], units="decision"); plt.show()
interactive(surface(z, exaggeration=2))

## Test sensitivity

If a small parameter change greatly alters the answer, a single threshold may communicate false precision. The heatmap displays the fraction for several assumptions while holding the data and validity mask fixed.

In [ ]:
heights = np.arange(1400, 1701, 50)
slopes = np.arange(6, 19, 2)
fractions = np.array([[(valid & (z > h) & (slope < s)).sum()/valid.sum() for h in heights] for s in slopes])
fig, ax = plt.subplots(figsize=(8, 4))
im = ax.imshow(fractions, origin="lower", aspect="auto", extent=(heights[0]-25, heights[-1]+25, slopes[0]-1, slopes[-1]+1), vmin=0, vmax=1)
ax.set(xlabel="Minimum elevation (m)", ylabel="Maximum slope (degrees)", title="Selected fraction under alternative thresholds")
fig.colorbar(im, ax=ax, label="Fraction of valid pixels"); plt.show()

## Export and verify

A reproducible deliverable contains both a machine-readable result and an explanation. Preserve nodata and test the file after writing. A successful numerical assertion does not replace a thoughtful interpretation.

In [ ]:
mask = np.where(valid, selected.astype(float), -9999).astype("float32")
path = OUTPUT / "capstone-selection.tif"
raster(mask).write(str(path), driver="COG")
np.testing.assert_array_equal(terrio.open_raster(str(path)).read(1), mask)
report = {"question": "Synthetic land above 1500 m with slope below 12 degrees",
          "source": CREDIT, "crs": CRS, "vertical_datum": "none: synthetic", "pixel_size_m": 30,
          "valid_pixels": int(valid.sum()), "selected_pixels": int(selected.sum()), "fraction": float(fraction),
          "area_km2": float(area_km2), "terrio_version": terrio.__version__,
          "limitations": ["Synthetic landscape", "Threshold-dependent result", "Projected grid area", "Missing slope edges excluded"],
          "citation": "Wu, Fitzgerald, and Morales Sanchez (2026), terrio, https://doi.org/10.5281/zenodo.23642204"}
(OUTPUT / "capstone-report.json").write_text(json.dumps(report, indent=2))
print("Deliverables:", path, OUTPUT / "capstone-report.json")

## Your investigation

Write a 200-word report with question, provenance, CRS/units, method, validity denominator, sensitivity, result, limitations, and TerrIO citation. Replace one threshold, rerun, and explain the difference.

### Discussion guide

A strong report states what was computed and what cannot be inferred. Download the notebook, report JSON, and GeoTIFF from JupyterLite to preserve your work outside browser storage.

## Sources and credit

- [Source 1](https://terrio.dev/citation/)
- [Source 2](https://terrio.dev/conformance/)
- [Source 3](https://doi.org/10.5281/zenodo.23642204)

TerrIO: Qiusheng Wu, Mavrick Fitzgerald, Guillermo Morales Sanchez, and contributors (2026). *terrio: Pure-Rust geospatial raster, vector and point cloud data I/O*, v0.9.0. [Zenodo concept DOI](https://doi.org/10.5281/zenodo.23642204). The DOI identifies the project; the execution version is pinned separately.

Natural Earth: Tom Patterson, Nathaniel Vaughn Kelso, and contributors, v5.1.2, [public-domain terms](https://www.naturalearthdata.com/about/terms-of-use/). Original explanations, synthetic data, code, and contour title card: JupyterLite-terrio contributors. Software and underlying third-party notices are documented in the repository's attribution page. Consult that page before redistributing dependency bundles.